In [7]:
DATA_DIR = "../data"

In [8]:
import pandas as pd

df = pd.read_csv(f"{DATA_DIR}/raw/dataset.csv")
df["date"] = pd.to_datetime(df["date"])

display(df)

,Unnamed: 0.1,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,...,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,0,0,2020-03-18,Recon 5,TeamOne,Dust2,0,16,2,2,...,1,0,15,5151,2340454,62,63,0,2,2
1,1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,...,6,5,10,5151,2340454,62,63,0,2,2
2,2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,...,6,3,10,5243,2340461,140,118,12,16,2
3,3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,...,8,7,8,5151,2340453,61,38,0,2,2
4,4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,...,5,4,11,5151,2340453,61,38,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45767,45767,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,...,7,5,9,1970,2299059,7,16,1,2,2
45768,45768,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,...,5,6,8,1970,2299059,7,16,1,2,2
45769,45769,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,...,8,9,4,1934,2299011,10,14,16,12,1
45770,45770,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,...,1,12,3,1934,2299001,6,12,16,4,1


In [9]:
# We sort the datased by increasing date as ELO computation needs to be done chronologically
df = df.sort_values("date").reset_index(drop=True)

display(df)

,Unnamed: 0.1,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,...,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,45771,45772,2015-11-03,NiP,Envy,Cobblestone,16,9,1,2,...,6,12,3,1934,2299003,6,1,16,9,1
1,45770,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,...,1,12,3,1934,2299001,6,12,16,4,1
2,45769,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,...,8,9,4,1934,2299011,10,14,16,12,1
3,45768,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,...,5,6,8,1970,2299059,7,16,1,2,2
4,45767,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,...,7,5,9,1970,2299059,7,16,1,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45767,4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,...,5,4,11,5151,2340453,61,38,0,2,2
45768,3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,...,8,7,8,5151,2340453,61,38,0,2,2
45769,2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,...,6,3,10,5243,2340461,140,118,12,16,2
45770,1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,...,6,5,10,5151,2340454,62,63,0,2,2


In [ ]:
import sys

def build_feature_state(df, base_elo=1500):
    """Build initial state from historical matches"""
    import pandas as pd  # type: ignore

    state = {
        "elo": {},
        "matches_played": {},
        "win_history": {},
        "h2h": {},
    }
    # Initialize Elo
    teams = pd.concat([df["team_1"], df["team_2"]]).unique()
    for team in teams:
        state["elo"][team] = base_elo
        state["matches_played"][team] = 0
        state["win_history"][team] = []
    return state

def compute_features_for_match(match, state):
    """Compute feature vector for a single match"""
    import numpy as np  # type: ignore

    team = match["team_1"]
    opp = match["team_2"]

    # Elo diff
    elo_team = state["elo"].get(team, 1500)
    elo_opp = state["elo"].get(opp, 1500)
    elo_diff = elo_team - elo_opp

    # Rolling winrates
    winrate_10 = (
        np.mean(state["win_history"].get(team, [])[-10:])
        if len(state["win_history"].get(team, [])) > 0
        else 0
    )
    winrate_30 = (
        np.mean(state["win_history"].get(team, [])[-30:])
        if len(state["win_history"].get(team, [])) > 0
        else 0
    )
    winrate_10_diff = winrate_10 - (
        np.mean(state["win_history"].get(opp, [])[-10:])
        if len(state["win_history"].get(opp, [])) > 0
        else 0
    )
    winrate_30_diff = winrate_30 - (
        np.mean(state["win_history"].get(opp, [])[-30:])
        if len(state["win_history"].get(opp, [])) > 0
        else 0
    )

    # Experience / matches played
    experience_diff = state["matches_played"].get(team, 0) - state[
        "matches_played"
    ].get(opp, 0)

    # Rank diff
    rank_diff = match["rank_1"] - match["rank_2"]

    # H2H winrate
    h2h_key = (team, opp)
    h2h_list = state["h2h"].get(h2h_key, [])
    h2h_winrate = np.mean(h2h_list) if len(h2h_list) > 0 else 0.5

    return {
        "elo_diff": elo_diff,
        "winrate_10_diff": winrate_10_diff,
        "winrate_30_diff": winrate_30_diff,
        "experience_diff": experience_diff,
        "rank_diff": rank_diff,
        "h2h_winrate": h2h_winrate,
    }

def update_state_with_result(match, state, k=32):
    """Update Elo, rolling winrates, H2H after a match"""
    team = match["team_1"]
    opp = match["team_2"]
    result = int(match["match_winner"] == 1)

    # Update Elo
    r_team = state["elo"].get(team, 1500)
    r_opp = state["elo"].get(opp, 1500)
    exp = 1 / (1 + 10 ** ((r_opp - r_team) / 400))
    state["elo"][team] = r_team + k * (result - exp)
    state["elo"][opp] = r_opp + k * ((1 - result) - (1 - exp))

    # Update win history
    state["win_history"].setdefault(team, []).append(result)
    state["win_history"].setdefault(opp, []).append(1 - result)

    # Update matches played
    state["matches_played"][team] = state["matches_played"].get(team, 0) + 1
    state["matches_played"][opp] = state["matches_played"].get(opp, 0) + 1

    # Update H2H
    h2h_key = (team, opp)
    state["h2h"].setdefault(h2h_key, []).append(result)
    return state

def match_to_features(preprocessor, history_df, match):
    """
    match: dict with team_1, team_2, rank_1, rank_2, date
    """
    import pandas as pd  # type: ignore

    # Build state from past only
    past = history_df[history_df["date"] < match["date"]]
    state = build_feature_state(past)

    # Replay past matches to update state
    for _, m in past.sort_values("date").iterrows():
        update_state_with_result(m, state)

    # Compute features for the future match
    X = pd.DataFrame([compute_features_for_match(match, state)])

    X_scaled = preprocessor.transform(X)

    return X_scaled

state = build_feature_state(df)
X = []
y = []

# Construct the features entirely from computed values, that's why X doesn't event concat the features with df
for _, match in df.iterrows():
    feats = compute_features_for_match(match, state)
    X.append(feats)
    y.append(int(match["match_winner"] == 1))
    state = update_state_with_result(match, state)

df_featured = pd.concat([df, pd.DataFrame(X), pd.Series(y, name="team_1_wins")], axis=1)

display(df_featured)

,Unnamed: 0.1,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,...,map_wins_1,map_wins_2,match_winner,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins
0,45771,45772,2015-11-03,NiP,Envy,Cobblestone,16,9,1,2,...,16,9,1,0.000000,0.0,0.000000,0,5,0.5,1
1,45770,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,...,16,4,1,16.000000,1.0,1.000000,1,-6,0.5,1
2,45769,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,...,16,12,1,0.000000,0.0,0.000000,0,-4,0.5,1
3,45768,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,...,1,2,2,0.000000,0.0,0.000000,0,-9,0.5,0
4,45767,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,...,1,2,2,-32.000000,-1.0,-1.000000,0,-9,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45767,4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,...,0,2,2,-77.312342,0.3,-0.166667,-18,23,0.0,0
45768,3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,...,0,2,2,-102.306861,0.3,-0.166667,-18,23,0.0,0
45769,2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,...,12,16,2,-36.844139,0.2,-0.157971,21,22,1.0,0
45770,1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,...,0,2,2,39.175377,0.2,0.133333,-491,-1,0.5,0


In [11]:
df_featured.to_csv(f"{DATA_DIR}/featured/results.csv", index=False)